In [2]:
import re
import fitz

def segment_document(text: str) -> dict:
    """
    Détecte les sections d'un CV par regex sur les headers.
    Retourne un dict {section_name: section_text}
    """
    section_patterns = [
        r"^(Education|EDUCATION|Academic Background)",
        r"^(Publication|PUBLICATION|Publications|PUBLICATIONS)",
        r"^(Experience|EXPERIENCE|Work Experience)",
        r"^(Skills?|SKILLS?|Skill Highlights)",
        r"^(Projects?|PROJECTS?)",
        r"^(Summary|SUMMARY|Profile|PROFILE)",
        r"^(Awards?|AWARDS?|Honors?)",
    ]

    lines = text.split('\n')
    sections = {}
    current_section = "HEADER"
    current_lines = []

    for line in lines:
        line_stripped = line.strip()
        matched = False
        for pattern in section_patterns:
            if re.match(pattern, line_stripped, re.IGNORECASE):
                if current_lines:
                    sections[current_section] = '\n'.join(current_lines)
                current_section = line_stripped.upper().split()[0]
                current_lines = []
                matched = True
                break
        if not matched:
            current_lines.append(line)

    if current_lines:
        sections[current_section] = '\n'.join(current_lines)

    return sections

# Test sur le CV d'Esra
doc = fitz.open("../data/raw/cvs/257264722 CV.pdf")
full_text = ""
for page in doc:
    full_text += page.get_text()

sections = segment_document(full_text)

print(f"Sections détectées : {list(sections.keys())}")
print(f"\n--- SECTION EDUCATION ---")
print(sections.get("EDUCATION", "NON TROUVÉE"))
print(f"\n--- SECTION PUBLICATION ---")
print(sections.get("PUBLICATION", "NON TROUVÉE"))


Sections détectées : ['HEADER', 'SUMMARY', 'PROJECT', 'SKILL', 'EDUCATION', 'EXPERIENCE', 'PUBLICATION', 'PROJECTS']

--- SECTION EDUCATION ---
 
Master of Science in Cybersecurity - Summer 2024 
New York University, New York 
 
Bachelor of Science in Software Engineering - Spring 2017 
AlHosn University, Abu Dhabi, UAE 
 
 

--- SECTION PUBLICATION ---
 
 Al-Nashash, E. J., & Olmsted, A. (2023). Enhancing IoT Security using Blockchain-Based 
Access Control Mechanisms and Database Management. In Proceedings of the 
International Conference for Internet Technology and Secured Transactions (ICITST-
2023), St Anne’s College, Oxford University, UK, 13-15 November 2023. DOI: 
10.20533/ICITST.2023.0021. ISBN: 978-1-913572-63-1. 
 


In [3]:
doc2 = fitz.open("../data/raw/transcripts/225127909 T.pdf")
full_text2 = ""
for page in doc2:
    full_text2 += page.get_text()

sections2 = segment_document(full_text2)

print(f"Sections détectées : {list(sections2.keys())}")
print(f"\n--- SECTION EDUCATION ---")
print(sections2.get("EDUCATION", "NON TROUVÉE"))
print(f"\n--- SECTION PUBLICATION ---")
print(sections2.get("PUBLICATION", "NON TROUVÉE"))


Sections détectées : ['HEADER']

--- SECTION EDUCATION ---
NON TROUVÉE

--- SECTION PUBLICATION ---
NON TROUVÉE


In [4]:
print(f"Caractères extraits par PyMuPDF : {len(full_text2)}")

Caractères extraits par PyMuPDF : 0


In [5]:
def detect_document_type(pdf_path: str) -> str:
    doc = fitz.open(pdf_path)
    total_chars = sum(len(page.get_text()) for page in doc)
    avg_chars_per_page = total_chars / len(doc)

    if avg_chars_per_page < 200:
        return "scanned_pdf"   # → route vers OCR
    else:
        return "native_pdf"    # → route vers PyMuPDF direct

In [6]:
cv_path = "../data/raw/cvs/257264722 CV.pdf"
transcript_path = "../data/raw/transcripts/225127909 T.pdf"

print(f"CV Esra      : {detect_document_type(cv_path)}")
print(f"Transcript   : {detect_document_type(transcript_path)}")


CV Esra      : native_pdf
Transcript   : scanned_pdf


# Testez l'import depuis le notebook

In [7]:
import sys
sys.path.insert(0, "..")
from src.ingestion.pdf_reader import PDFReader

reader = PDFReader()

cv_result = reader.extract("../data/raw/cvs/257264722 CV.pdf")
print(f"CV      : {cv_result.document_type} — {cv_result.char_count} chars")

tr_result = reader.extract("../data/raw/transcripts/225127909 T.pdf")
print(f"Transcript : {tr_result.document_type} — needs_ocr={tr_result.needs_ocr}")


CV      : native_pdf — 3208 chars
Transcript : scanned_pdf — needs_ocr=True


# Testez la segmentation depuis le notebook

In [8]:
import sys
sys.path.insert(0, "..")
from src.preprocessing.segmenter import segment_document
import fitz

# Test CV natif
doc = fitz.open("../data/raw/cvs/257264722 CV.pdf")
text = "".join(page.get_text() for page in doc)

result = segment_document(text)
print(f"Sections    : {result.detected_section_names}")
print(f"Confidence  : {result.confidence}")
print(f"Warning     : {result.warning}")
print(f"\n--- EDUCATION ---")
print(result.sections.get("EDUCATION", "NON TROUVÉE"))
print(f"\n--- PUBLICATIONS ---")
print(result.sections.get("PUBLICATIONS", "NON TROUVÉE"))


Sections    : ['HEADER', 'SUMMARY', 'SKILLS', 'EDUCATION', 'EXPERIENCE', 'PUBLICATIONS', 'PROJECTS']
Confidence  : 1.0

--- EDUCATION ---
Master of Science in Cybersecurity - Summer 2024
New York University, New York

Bachelor of Science in Software Engineering - Spring 2017
AlHosn University, Abu Dhabi, UAE

--- PUBLICATIONS ---
 Al-Nashash, E. J., & Olmsted, A. (2023). Enhancing IoT Security using Blockchain-Based
Access Control Mechanisms and Database Management. In Proceedings of the
International Conference for Internet Technology and Secured Transactions (ICITST-
2023), St Anne’s College, Oxford University, UK, 13-15 November 2023. DOI:
10.20533/ICITST.2023.0021. ISBN: 978-1-913572-63-1.
